# V23: larger-data contrastive replacement component

Trains our qualified v22 external-only checkpoint on all host prompts, then fits v19-style multilayer readouts. This notebook makes no validation claim and creates no submission. Separate inference adds the component to the credited Nishant Dahal v16 ensemble. LoRA: Hu et al.; SupCon: Khosla et al. Public pool credited to Nishant Dahal and original contributors, CC BY-SA 4.0.

In [ ]:

import os,sys,json,glob,hashlib,subprocess,importlib.metadata
from pathlib import Path
from packaging.version import Version
for name in ('OMP_NUM_THREADS','OPENBLAS_NUM_THREADS','MKL_NUM_THREADS'): os.environ[name]='2'
os.environ['HF_HUB_OFFLINE']='1';os.environ['TRANSFORMERS_OFFLINE']='1';os.environ['TOKENIZERS_PARALLELISM']='false'
try: version=importlib.metadata.version('torchao')
except importlib.metadata.PackageNotFoundError: version=None
if version is not None and Version(version)<Version('0.16.0'):
    subprocess.run([sys.executable,'-m','pip','uninstall','-y','torchao'],check=True)
import torch,peft,transformers
assert torch.cuda.device_count()>=1
print({'torch':torch.__version__,'peft':peft.__version__,'transformers':transformers.__version__},flush=True)
plan=json.loads("{\"experiment\": \"improvement-v23\", \"kind\": \"larger-data replacement of the v21 additive component\", \"incumbent\": \"adithyagiri/name-that-llm-contrastive-blend-v21\", \"incumbent_public_score\": 0.622, \"initialization\": \"v22 ce_supcon external_only adapter and heads, trained on 18000 external passages; no host-adapted checkpoint\", \"training\": \"All 2699 host rows, exactly two equivalent epochs with original v18 host-stage objective, paired sampler and optimizer\", \"readout\": {\"layers\": [12, 24, 36], \"C\": 0.02, \"max_iter\": 1000, \"seed\": 1901, \"batch_size\": 8, \"fit_dtype\": \"float64 for portable numerical equivalence\"}, \"resume_state\": \"Restore adapter and all three heads; fresh host optimizer and GradScaler, since v18 did not save optimizer/scaler state\", \"blend\": {\"incumbent_members_unchanged\": true, \"new_member_log_probability_weight\": 0.15, \"position\": \"Before unchanged incumbent Sinkhorn\", \"replaces\": \"v21 contrastive component; same 0.15 vote, not an additional correlated vote\"}, \"release_policy\": \"One prospective submission after complete all-host training, finite real inference for all visible rows, ordered label mapping and schema checks. No claim of ensemble CV improvement; retain v16 as incumbent unless scored improvement.\", \"reason\": \"V22 contrastive readout 0.4425925926 and fixed sparse blend 0.5111111111 exceed frozen qualification thresholds; require full artifact audit before building\", \"confirmation_status\": \"V22 fixed pilot qualifies; five-fold v20 remains unrun; no full-ensemble CV claim\", \"attribution\": \"Credited Nishant Dahal v16 ensemble retained. Our v23 replaces the v21 contrastive readout using the larger v22 external checkpoint. Established LoRA/SupCon methods and original public-data contributors retain credit.\", \"limitations\": [\"Same repeatedly inspected fold; two-arm selection reported\", \"Readout and sparse-blend improvements are not complete-ensemble validation\", \"Public score may fall\", \"Existing v16 test-distribution assumptions retained\"], \"packaging_template_sha256\": \"78f9349d783abbc873367e716f0e2e58e2a0d8bef394735cc4638a8757fafdbd\", \"training_parameters\": {\"seed\": 1801, \"max_length\": 256, \"batch_size\": 4, \"gradient_accumulation\": 8, \"lora_rank\": 16, \"lora_alpha\": 32, \"target_modules\": [\"q_proj\", \"v_proj\"], \"host_epochs\": 2, \"host_adapter_lr\": 3e-05, \"host_head_lr\": 0.0001, \"contrastive_weight\": 0.1, \"contrastive_temperature\": 0.1, \"wall_limit_seconds\": 25200}, \"labels\": [\"claude-fable-5\", \"claude-opus-4-6\", \"claude-opus-4-8\", \"claude-opus-5\", \"claude-sonnet-5\", \"gemini-3.1-pro-preview\", \"gemini-3.6-flash\", \"gemini-3.7-flash\", \"gemini-3.8-flash\", \"glm-5.3\", \"glm-5.3-flash\", \"gpt-5.5\", \"gpt-5.6-luna\", \"gpt-5.6-sol\", \"gpt-5.6-terra\", \"gpt-6-astra\", \"kimi-k3\", \"qwen3.8-max\"], \"host_sha256\": \"40a22ce8d88c5f46feb4637a099db815bef5f713ffe53e89dbe1b7b21de77707\", \"model\": \"qwen-lm/qwen-3/Transformers/4b-base/1\", \"checkpoint_sha256\": {\"adapter/adapter_model.safetensors\": \"d207ce5a44b6fea2a2008fc398e81bddfa4e8f1f13b7b9d9c0527077d693f66a\", \"adapter/adapter_config.json\": \"5cc84f15952fcb3d9c192cc8d819fc201a31c73f95884656c5fe2acabda33ab7\", \"heads.pt\": \"3f9cf4f10a2cbf97001803cd8b1794d01623aee39583dc332104fa5aa346fd4a\"}, \"source_sha256\": {\"v18_worker.py\": \"a79591118969d7e2b687523a64c46f68399a217c36972d37e300aaba7d440b34\", \"train_full.py\": \"b8bc0cc2393fc0412f9bbeb0952e8d55ccecad60e48411ab8846ef7b060e47df\", \"component_v21.py\": \"820fc727fd549736b97df9888dc51065fdc89a653c8ac90d689e370aa244c417\"}}")
matches=[Path(p).parent for p in glob.glob('/kaggle/input/**/plan.json',recursive=True)
         if json.loads(Path(p).read_text()).get('experiment')=='improvement-v22']
assert len(matches)==1,matches
plan['checkpoint']=str(matches[0]/'ce_supcon/external_only')
paths=[p for p in glob.glob('/kaggle/input/**/config.json',recursive=True) if '/4b-base/' in p.lower()]
assert len(paths)==1,paths
plan['model_dir']=str(Path(paths[0]).parent)
plan['host_path']=next(p for p in glob.glob('/kaggle/input/**/train.csv',recursive=True) if 'name-that-llm' in p)
assert hashlib.sha256(Path(plan['host_path']).read_bytes()).hexdigest()==plan['host_sha256']
Path('plan.json').write_text(json.dumps(plan,indent=2))
sources={'v18_worker.py': '"""Standalone GPU worker for the frozen v18 paired-objective pilot."""\nimport json\nimport math\nimport os\nfrom pathlib import Path\nimport time\n\nimport numpy as np\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\ndef supcon_loss(features, labels, temperature=0.1):\n    """Mean log-probability of same-class positives, excluding self pairs."""\n    z = F.normalize(features.float(), dim=1)\n    logits = z @ z.T / temperature\n    diagonal = torch.eye(len(z), dtype=torch.bool, device=z.device)\n    positives = labels[:, None].eq(labels[None, :]) & ~diagonal\n    counts = positives.sum(1)\n    if not bool((counts > 0).all()):\n        raise ValueError("Every contrastive anchor must have a distinct positive.")\n    denominator = torch.logsumexp(logits.masked_fill(diagonal, -torch.inf), dim=1)\n    return -((logits - denominator[:, None]).masked_fill(~positives, 0).sum(1) / counts).mean()\n\n\ndef paired_batches(frame, steps, seed):\n    """Two authors, two different prompts each; identical schedule for both arms."""\n    rng = np.random.default_rng(seed)\n    groups = {label: {p: sub.index.to_numpy() for p, sub in group.groupby("group")}\n              for label, group in frame.groupby("label")}\n    assert all(len(g) >= 2 for g in groups.values())\n    labels = sorted(groups)\n    for _ in range(steps):\n        selected = []\n        for label in rng.choice(labels, 2, replace=False):\n            for prompt in rng.choice(list(groups[label]), 2, replace=False):\n                selected.append(int(rng.choice(groups[label][prompt])))\n        yield selected\n\n\nclass AttributionModel(nn.Module):\n    def __init__(self, model_dir, plan):\n        super().__init__()\n        from peft import LoraConfig, get_peft_model\n        from transformers import AutoModel\n        backbone = AutoModel.from_pretrained(model_dir, torch_dtype=torch.float16,\n                                            local_files_only=True, attn_implementation="sdpa")\n        backbone.config.use_cache = False\n        backbone.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})\n        self.encoder = get_peft_model(backbone, LoraConfig(r=plan["lora_rank"],\n            lora_alpha=plan["lora_alpha"], target_modules=plan["target_modules"], lora_dropout=0.0,\n            bias="none"))\n        width = backbone.config.hidden_size\n        self.norm = nn.LayerNorm(width)\n        self.head = nn.Linear(width, 18)\n        self.projection = nn.Sequential(nn.Linear(width, 256), nn.GELU(), nn.Linear(256, 128))\n\n    def forward(self, batch):\n        hidden = self.encoder(**batch, use_cache=False).last_hidden_state.float()\n        mask = batch["attention_mask"].clone()\n        mask[:, 0] = 0\n        pooled = (hidden * mask.unsqueeze(-1)).sum(1) / mask.sum(1, keepdim=True).clamp_min(1)\n        features = self.norm(pooled)\n        return self.head(features), self.projection(features)\n\n\ndef main():\n    import pandas as pd\n    from transformers import AutoTokenizer\n    import transformers, peft\n    plan = json.loads(Path("plan.json").read_text())\n    variant = os.environ["V18_VARIANT"]\n    assert variant in plan["variants"]\n    out = Path(variant)\n    out.mkdir(exist_ok=True)\n    started = time.monotonic()\n    def log(**kw):\n        record = dict(variant=variant, seconds=round(time.monotonic()-started, 2), **kw)\n        print(json.dumps(record), flush=True)\n        with (out / "progress.jsonl").open("a") as f:\n            f.write(json.dumps(record)+"\\n")\n    assert torch.cuda.device_count() == 1\n    torch.manual_seed(plan["seed"])\n    np.random.seed(plan["seed"])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    model_dir = plan["model_dir"]\n    tok = AutoTokenizer.from_pretrained(model_dir, local_files_only=True)\n    tok.padding_side = "right"\n    if tok.pad_token_id is None:\n        tok.pad_token = tok.eos_token\n    frame = pd.read_parquet("training_frame.parquet")\n    valid = pd.read_parquet("validation_frame.parquet")\n    assert set(frame.loc[frame.stage == "host", "group"]).isdisjoint(set(valid.group))\n    def tokenize(df):\n        enc = tok(df.text.tolist(), truncation=True, max_length=plan["max_length"]-1,\n                  add_special_tokens=False)\n        return [ids+[tok.eos_token_id] for ids in enc["input_ids"]]\n    tokens, val_tokens = tokenize(frame), tokenize(valid)\n    def batch(ids, bank):\n        return {k: v.cuda() for k,v in tok.pad({"input_ids": [bank[i] for i in ids]},\n                                              padding=True, return_tensors="pt").items()}\n    model = AttributionModel(model_dir, plan).cuda()\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    # Frozen base is fp16; trainable adapters and heads must be fp32 for GradScaler.\n    for p in trainable:\n        p.data = p.data.float()\n    log(event="loaded", trainable_parameters=sum(p.numel() for p in trainable),\n        torch=torch.__version__, transformers=transformers.__version__, peft=peft.__version__,\n        gpu=torch.cuda.get_device_name(0), training_rows=len(frame), validation_rows=len(valid))\n    scaler = torch.amp.GradScaler("cuda")\n    skipped = 0\n    for stage, epochs in [("pool", plan["pool_epochs"]), ("host", plan["host_epochs"])]:\n        subset = frame.loc[frame.stage == stage]\n        micro_steps = math.ceil(len(subset)/plan["batch_size"])\n        optimizer = torch.optim.AdamW([\n            {"params": [p for p in model.encoder.parameters() if p.requires_grad], "lr": plan[stage+"_adapter_lr"]},\n            {"params": list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\n             "lr": plan[stage+"_head_lr"]}], weight_decay=0.01)\n        for epoch in range(epochs):\n            model.train()\n            optimizer.zero_grad(set_to_none=True)\n            losses = []\n            for step, ids in enumerate(paired_batches(subset, micro_steps, plan["seed"] + epoch + (100 if stage == "host" else 0))):\n                if time.monotonic()-started > plan["wall_limit_seconds"]:\n                    raise TimeoutError("Frozen pilot wall budget exhausted; no complete result claimed")\n                labels = torch.as_tensor(frame.loc[ids, "label"].to_numpy(), device="cuda")\n                with torch.autocast("cuda", dtype=torch.float16):\n                    logits, features = model(batch(ids, tokens))\n                    ce = F.cross_entropy(logits.float(), labels)\n                    contrastive = supcon_loss(features, labels, plan["contrastive_temperature"])\n                    loss = ce + (plan["contrastive_weight"] * contrastive if variant == "ce_supcon" else 0)\n                if not torch.isfinite(loss):\n                    raise FloatingPointError("Nonfinite training loss")\n                window_start = (step // plan["gradient_accumulation"]) * plan["gradient_accumulation"]\n                divisor = min(plan["gradient_accumulation"], micro_steps-window_start)\n                scaler.scale(loss/divisor).backward()\n                losses.append(float(loss.detach()))\n                if (step+1) % plan["gradient_accumulation"] == 0 or step+1 == micro_steps:\n                    scaler.unscale_(optimizer)\n                    torch.nn.utils.clip_grad_norm_(trainable, 1.0)\n                    before = scaler.get_scale()\n                    scaler.step(optimizer)\n                    scaler.update()\n                    skipped += int(scaler.get_scale() < before)\n                    optimizer.zero_grad(set_to_none=True)\n                if step % 50 == 0:\n                    log(event="train", stage=stage, epoch=epoch, step=step, steps=micro_steps,\n                        mean_loss=float(np.mean(losses[-50:])), ce=float(ce.detach()),\n                        contrastive=float(contrastive.detach()), skipped_steps=skipped,\n                        peak_gb=torch.cuda.max_memory_allocated()/2**30)\n            log(event="epoch_complete", stage=stage, epoch=epoch, mean_loss=float(np.mean(losses)))\n        checkpoint = out / ("external_only" if stage == "pool" else "final")\n        model.encoder.save_pretrained(checkpoint / "adapter")\n        torch.save({"norm": model.norm.state_dict(), "head": model.head.state_dict(),\n                    "projection": model.projection.state_dict()}, checkpoint / "heads.pt")\n    model.eval()\n    scores = []\n    with torch.inference_mode():\n        for start in range(0, len(valid), 4):\n            ids = list(range(start, min(start+4, len(valid))))\n            with torch.autocast("cuda", dtype=torch.float16):\n                logits, _ = model(batch(ids, val_tokens))\n            scores.append(logits.float().cpu().numpy())\n    scores = np.concatenate(scores)\n    assert np.isfinite(scores).all()\n    np.savez_compressed(out / "validation.npz", ids=valid.id.to_numpy(dtype=str),\n                        labels=np.asarray(plan["labels"], dtype=str), logits=scores,\n                        targets=valid.label.to_numpy())\n    result = {"variant":variant, "accuracy":float((scores.argmax(1)==valid.label.to_numpy()).mean()),\n              "seconds":time.monotonic()-started, "skipped_steps":skipped,\n              "validation_rows":len(valid), "completed":True}\n    (out / "result.json").write_text(json.dumps(result, indent=2))\n    log(event="complete", result=result)\n\n\nif __name__ == "__main__":\n    main()\n', 'train_full.py': '"""Resume our external-only checkpoint on every host prompt, with fixed training."""\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path\nimport time\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.nn import functional as F\nfrom peft import set_peft_model_state_dict\nfrom safetensors.torch import load_file\nfrom transformers import AutoTokenizer\nfrom v18_worker import AttributionModel, paired_batches, supcon_loss\n\n\ndef main():\n    plan = json.loads(Path(\'plan.json\').read_text())\n    params = plan[\'training_parameters\']\n    out = Path(\'full\'); out.mkdir(exist_ok=True)\n    started = time.monotonic()\n    def log(**record):\n        record[\'seconds\'] = round(time.monotonic()-started, 2)\n        print(json.dumps(record), flush=True)\n        with (out/\'progress.jsonl\').open(\'a\') as f: f.write(json.dumps(record)+\'\\n\')\n    torch.manual_seed(params[\'seed\']); np.random.seed(params[\'seed\'])\n    torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction = False\n    assert torch.cuda.device_count() == 1\n    frame = pd.read_csv(plan[\'host_path\'], dtype={\'id\':str})\n    assert len(frame)==2699 and frame.id.is_unique and frame.prompt_id.nunique()==150\n    assert hashlib.sha256(Path(plan[\'host_path\']).read_bytes()).hexdigest()==plan[\'host_sha256\']\n    labels = np.asarray(plan[\'labels\'])\n    assert np.array_equal(labels, np.sort(frame.model.unique()))\n    frame[\'label\'] = frame.model.map({v:i for i,v in enumerate(labels)})\n    frame[\'group\'] = frame.prompt_id\n    assert frame.label.notna().all()\n    tokenizer = AutoTokenizer.from_pretrained(plan[\'model_dir\'], local_files_only=True)\n    tokenizer.padding_side = \'right\'\n    if tokenizer.pad_token_id is None: tokenizer.pad_token = tokenizer.eos_token\n    tokens = tokenizer(frame.text.tolist(), truncation=True, max_length=params[\'max_length\']-1,\n                       add_special_tokens=False)[\'input_ids\']\n    tokens = [ids+[tokenizer.eos_token_id] for ids in tokens]\n    model = AttributionModel(plan[\'model_dir\'], params)\n    checkpoint = Path(plan[\'checkpoint\'])\n    for relative, digest in plan[\'checkpoint_sha256\'].items():\n        assert hashlib.sha256((checkpoint/relative).read_bytes()).hexdigest()==digest\n    state = load_file(str(checkpoint/\'adapter/adapter_model.safetensors\'))\n    result = set_peft_model_state_dict(model.encoder, state)\n    assert not result.unexpected_keys, result.unexpected_keys\n    assert not [k for k in result.missing_keys if \'lora_\' in k], result.missing_keys\n    heads = torch.load(checkpoint/\'heads.pt\', map_location=\'cpu\', weights_only=True)\n    for name in [\'norm\',\'head\',\'projection\']: getattr(model,name).load_state_dict(heads[name], strict=True)\n    model = model.cuda()\n    trainable = [p for p in model.parameters() if p.requires_grad]\n    for p in trainable: p.data = p.data.float()\n    initial_adapter = {n:p.detach().cpu().clone() for n,p in model.encoder.named_parameters() if p.requires_grad}\n    optimizer = torch.optim.AdamW([\n        {\'params\':[p for p in model.encoder.parameters() if p.requires_grad], \'lr\':params[\'host_adapter_lr\']},\n        {\'params\':list(model.norm.parameters())+list(model.head.parameters())+list(model.projection.parameters()),\n         \'lr\':params[\'host_head_lr\']}], weight_decay=0.01)\n    scaler = torch.amp.GradScaler(\'cuda\'); skipped = 0\n    steps = math.ceil(len(frame)/params[\'batch_size\'])\n    log(event=\'loaded\', rows=len(frame), prompts=frame.prompt_id.nunique(), steps_per_epoch=steps,\n        trainable_parameters=sum(p.numel() for p in trainable))\n    for epoch in range(params[\'host_epochs\']):\n        model.train(); optimizer.zero_grad(set_to_none=True); losses=[]\n        for step,ids in enumerate(paired_batches(frame, steps, params[\'seed\']+100+epoch)):\n            if time.monotonic()-started>params[\'wall_limit_seconds\']: raise TimeoutError(\'Training budget exhausted\')\n            batch = {k:v.cuda() for k,v in tokenizer.pad({\'input_ids\':[tokens[i] for i in ids]},\n                                                       padding=True, return_tensors=\'pt\').items()}\n            targets = torch.as_tensor(frame.loc[ids,\'label\'].to_numpy(), device=\'cuda\')\n            with torch.autocast(\'cuda\', dtype=torch.float16):\n                logits, features = model(batch)\n                ce = F.cross_entropy(logits.float(), targets)\n                contrastive = supcon_loss(features, targets, params[\'contrastive_temperature\'])\n                loss = ce+params[\'contrastive_weight\']*contrastive\n            if not torch.isfinite(loss): raise FloatingPointError(\'Nonfinite loss\')\n            window_start=(step//params[\'gradient_accumulation\'])*params[\'gradient_accumulation\']\n            divisor=min(params[\'gradient_accumulation\'],steps-window_start)\n            scaler.scale(loss/divisor).backward(); losses.append(float(loss.detach()))\n            if (step+1)%params[\'gradient_accumulation\']==0 or step+1==steps:\n                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(trainable,1.0)\n                before=scaler.get_scale(); scaler.step(optimizer); scaler.update()\n                skipped+=int(scaler.get_scale()<before); optimizer.zero_grad(set_to_none=True)\n            if step%50==0:\n                log(event=\'train\',epoch=epoch,step=step,steps=steps,mean_loss=float(np.mean(losses[-50:])),\n                    ce=float(ce.detach()),contrastive=float(contrastive.detach()),skipped_steps=skipped)\n        log(event=\'epoch_complete\',epoch=epoch,mean_loss=float(np.mean(losses)))\n    assert all(torch.isfinite(p).all() for p in trainable)\n    changed=sum(not torch.equal(initial_adapter[n],p.detach().cpu()) for n,p in model.encoder.named_parameters() if p.requires_grad)\n    assert changed>0\n    model.encoder.save_pretrained(out/\'adapter\')\n    torch.save({n:getattr(model,n).state_dict() for n in [\'norm\',\'head\',\'projection\']}, out/\'heads.pt\')\n    result={\'completed\':True,\'rows\':len(frame),\'prompts\':frame.prompt_id.nunique(),\n            \'epochs\':params[\'host_epochs\'],\'skipped_steps\':skipped,\'changed_adapter_tensors\':changed,\n            \'seconds\':time.monotonic()-started,\'labels\':labels.tolist()}\n    (out/\'training_result.json\').write_text(json.dumps(result,indent=2))\n    log(event=\'complete\',result=result)\n\n\nif __name__==\'__main__\': main()\n', 'component_v21.py': '"""Feature extraction and portable logistic readout for the new v21 member."""\nimport gc\nimport json\nfrom pathlib import Path\nimport time\nimport warnings\nimport numpy as np\nimport torch\nfrom scipy.special import log_softmax\n\n\nclass ContrastiveReadout:\n    def __init__(self, model_dir, adapter, layers=(12,24,36), max_length=256, batch_size=8):\n        self.model_dir, self.adapter = model_dir, adapter\n        self.layers, self.max_length, self.batch_size = list(layers), max_length, batch_size\n\n    def features(self, texts):\n        from transformers import AutoModel, AutoTokenizer\n        from peft import PeftModel\n        tokenizer = AutoTokenizer.from_pretrained(self.model_dir,local_files_only=True)\n        tokenizer.padding_side=\'right\'\n        if tokenizer.pad_token_id is None: tokenizer.pad_token=tokenizer.eos_token\n        tokens=tokenizer(list(texts),truncation=True,max_length=self.max_length-1,\n                         add_special_tokens=False)[\'input_ids\']\n        tokens=[ids+[tokenizer.eos_token_id] for ids in tokens]\n        order=np.argsort([len(ids) for ids in tokens],kind=\'stable\')\n        torch.backends.cuda.matmul.allow_fp16_reduced_precision_reduction=False\n        model=AutoModel.from_pretrained(self.model_dir,torch_dtype=torch.float16,\n                                        attn_implementation=\'sdpa\',local_files_only=True)\n        model=PeftModel.from_pretrained(model,self.adapter,is_trainable=False).cuda().eval()\n        features=np.empty((len(texts),len(self.layers),model.config.hidden_size),dtype=np.float32)\n        started=time.monotonic()\n        with torch.inference_mode():\n            for start in range(0,len(texts),self.batch_size):\n                ids=order[start:start+self.batch_size]\n                batch={k:v.cuda() for k,v in tokenizer.pad({\'input_ids\':[tokens[i] for i in ids]},\n                                                          padding=True,return_tensors=\'pt\').items()}\n                with torch.autocast(\'cuda\',dtype=torch.float16):\n                    states=model(**batch,output_hidden_states=True,use_cache=False).hidden_states\n                mask=batch[\'attention_mask\'].clone();mask[:,0]=0\n                for j,layer in enumerate(self.layers):\n                    h=states[layer].float()\n                    features[ids,j]=((h*mask.unsqueeze(-1)).sum(1)/mask.sum(1,keepdim=True).clamp_min(1)).cpu().numpy()\n                del states\n                if start%400==0:\n                    print(json.dumps({\'event\':\'v21_features\',\'completed\':min(start+len(ids),len(texts)),\n                                      \'total\':len(texts),\'seconds\':time.monotonic()-started}),flush=True)\n        del model;gc.collect();torch.cuda.empty_cache()\n        assert np.isfinite(features).all()\n        return features\n\n    @staticmethod\n    def predict(features, path):\n        with np.load(path,allow_pickle=False) as state:\n            assert features.shape[1:]==(len(state[\'layers\']),state[\'mean\'].shape[-1])\n            scores=[]\n            for j in range(features.shape[1]):\n                X=(features[:,j].astype(np.float64)-state[\'mean\'][j])/state[\'scale\'][j]\n                scores.append(log_softmax(X@state[\'coef\'][j].T+state[\'intercept\'][j],axis=1))\n            labels=state[\'labels\'].copy()\n        scores=np.mean(scores,axis=0)\n        assert np.isfinite(scores).all()\n        return labels,scores\n\n    def fit(self,features,targets,path,C=0.02,max_iter=1000,seed=1901):\n        from sklearn.preprocessing import StandardScaler\n        from sklearn.linear_model import LogisticRegression\n        from sklearn.exceptions import ConvergenceWarning\n        means,scales,coefs,intercepts,predictions=[],[],[],[],[]\n        labels=np.sort(np.unique(targets))\n        for j,layer in enumerate(self.layers):\n            scaler=StandardScaler()\n            X=scaler.fit_transform(features[:,j].astype(np.float64))\n            clf=LogisticRegression(C=C,max_iter=max_iter,random_state=seed)\n            with warnings.catch_warnings():\n                warnings.simplefilter(\'error\',ConvergenceWarning)\n                clf.fit(X,targets)\n            assert np.array_equal(labels,clf.classes_)\n            means.append(scaler.mean_);scales.append(scaler.scale_)\n            coefs.append(clf.coef_);intercepts.append(clf.intercept_)\n            predictions.append(clf.predict_log_proba(X))\n        np.savez_compressed(path,labels=labels.astype(str),layers=self.layers,mean=np.stack(means),\n                            scale=np.stack(scales),coef=np.stack(coefs),intercept=np.stack(intercepts))\n        recovered_labels,recovered_scores=self.predict(features,path)\n        assert np.array_equal(recovered_labels,labels)\n        np.testing.assert_allclose(recovered_scores,np.mean(predictions,axis=0),atol=1e-8,rtol=1e-8)\n        return {\'fit_rows\':len(targets),\'layers\':self.layers,\'labels\':labels.tolist(),\n                \'portable_readout_check\':\'passed\',\'C\':C,\'dtype\':\'float64 for scaler and LR\'}\n\n\ndef main():\n    import hashlib\n    import pandas as pd\n    plan=json.loads(Path(\'plan.json\').read_text())\n    frame=pd.read_csv(plan[\'host_path\'],dtype={\'id\':str})\n    component=ContrastiveReadout(plan[\'model_dir\'],\'full/adapter\',layers=plan[\'readout\'][\'layers\'],\n                                max_length=plan[\'training_parameters\'][\'max_length\'],\n                                batch_size=plan[\'readout\'][\'batch_size\'])\n    features=component.features(frame.text.tolist())\n    np.save(\'full/train_features.npy\',features)\n    result=component.fit(features,frame.model.to_numpy(),\'full/readout.npz\',\n                          C=plan[\'readout\'][\'C\'],max_iter=plan[\'readout\'][\'max_iter\'],seed=plan[\'readout\'][\'seed\'])\n    result[\'artifacts_sha256\']={str(p):hashlib.sha256(p.read_bytes()).hexdigest()\n                                for p in [Path(\'full/adapter/adapter_model.safetensors\'),Path(\'full/readout.npz\'),Path(\'full/train_features.npy\')]}\n    Path(\'full/readout_result.json\').write_text(json.dumps(result,indent=2))\n    print(json.dumps(result),flush=True)\n\n\nif __name__==\'__main__\': main()\n'}
for name,source in sources.items():
    assert hashlib.sha256(source.encode()).hexdigest()==plan['source_sha256'][name]
    Path(name).write_text(source)
env=dict(os.environ,CUDA_VISIBLE_DEVICES='0')
subprocess.run([sys.executable,'-u','train_full.py'],env=env,check=True)
subprocess.run([sys.executable,'-u','component_v21.py'],env=env,check=True)
assert json.loads(Path('full/training_result.json').read_text())['completed']
assert Path('full/readout.npz').is_file()
print({'v23_training':'COMPLETE','submission_created':False},flush=True)
